In [0]:
%run /Workspace/Users/rahulpatel@cyntexa.com/IAP_Practice/voltpath/00_setup

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.user_entitlements (
    user_email    STRING,
    allowed_city  STRING,
    can_view_pii  BOOLEAN
) USING DELTA
""")

# Insert/update current user's row (can_view_pii = true so owner sees real phones)
spark.sql(f"""
MERGE INTO {catalog}.ops.user_entitlements AS t
USING (SELECT current_user() AS user_email, "ALL" AS allowed_city, true AS can_view_pii) AS s
ON t.user_email = s.user_email
WHEN MATCHED THEN UPDATE SET t.allowed_city = s.allowed_city, t.can_view_pii = s.can_view_pii
WHEN NOT MATCHED THEN INSERT (user_email, allowed_city, can_view_pii) VALUES (s.user_email, s.allowed_city, s.can_view_pii)
""")

display(spark.table(f"{catalog}.ops.user_entitlements"))

# --- G1: Grants to fk_analyst only ---
# USE CATALOG on catalog, USE SCHEMA + SELECT on gold schema, nothing else
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `fk_analyst`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.gold TO `fk_analyst`")
spark.sql(f"GRANT SELECT ON SCHEMA {catalog}.gold TO `fk_analyst`")


# --- G2: Mask phone in gold.dim_member ---
# Real value if can_view_pii = TRUE, otherwise XXXXXX + last 4 digits
# Function is self-contained: looks up the current user's can_view_pii internally
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.ops.mask_phone(phone_col STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member("fk_admins") THEN phone_col
    WHEN EXISTS (
        SELECT 1 FROM {catalog}.ops.user_entitlements e
        WHERE lower(e.user_email) = lower(current_user())
        AND e.can_view_pii = true
        ) THEN phone_col
    ELSE
        CONCAT(
          SUBSTR(phone_col, 1, 2),
          REPEAT('*', GREATEST(LENGTH(phone_col) - 4, 1)),
          SUBSTR(phone_col, -2)
        )
  END;
""")

spark.sql(f"""
ALTER TABLE {catalog}.gold.dim_member
    ALTER COLUMN phone SET MASK {catalog}.ops.mask_phone
""")

# print("Phone mask applied to gold.dim_member")
# display(spark.sql(f"SELECT member_sk, member_id, phone FROM {catalog}.gold.dim_member LIMIT 5"))

In [0]:

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.ops.daily_station_row_filter(city_col STRING)
RETURNS BOOLEAN
RETURN
  is_account_group_member('fk_admins')
  OR EXISTS (
    SELECT 1 FROM {catalog}.ops.user_entitlements e
    WHERE lower(e.user_email) = lower(current_user())
      AND (e.allowed_city = city_col OR e.allowed_city = 'ALL')
  );
""")

spark.sql(f"""
ALTER TABLE {catalog}.gold.agg_daily_station
  SET ROW FILTER {catalog}.ops.daily_station_row_filter ON (city);
""")

In [0]:
display(spark.sql(f"SELECT * FROM {catalog}.gold.agg_daily_station"))